# Лабораторная работа 7 — Метод Вольфа (Wolfe)

В этом ноутбуке реализован поиск по направлению, удовлетворяющий условиям Вольфа (Wolfe conditions), а также метод градиентного спуска с этим условием выбора шага. Также включён код симплекс-метода из Lab_4 для удобства (если потребуется).

Все пояснения и комментарии даны на русском языке.

In [ ]:
import numpy as np


def simplex_main_phase(c, A, x_init, B_init, max_iter=1000, forbidden_pairs=None):  # [WOLF] added forbidden_pairs
    x = np.array(x_init, dtype=float)
    B = list(B_init)

    if forbidden_pairs is None:
        forbidden_pairs = []

    for iteration in range(max_iter):
        A_B = A[:, B]
        try:
            A_B_inv = np.linalg.inv(A_B)
        except np.linalg.LinAlgError:
            raise ValueError("Матрица A_B вырождена. B не является базисом.")

        c_B = c[B]
        u = c_B @ A_B_inv
        delta = c - u @ A

        eps = 1e-9
        if np.all(delta <= eps):
            return "optimal", x, B

        # modification
        j0 = -1
        for j in range(len(delta)):
            if delta[j] > eps:
                
                forbidden = any(
                    (j == a and b in B) or (j == b and a in B)
                    for a, b in forbidden_pairs
                )
                if not forbidden:
                    j0 = j
                    break

        if j0 == -1:
            return "optimal", x, B
        # modification end

        z = A_B_inv @ A[:, j0]

        theta = np.full(len(B), np.inf)
        for i in range(len(B)):
            if z[i] > eps:
                theta[i] = x[B[i]] / z[i]

        theta0 = np.min(theta)

        if np.isinf(theta0):
            return "unbounded", None, B

        k = np.argmin(theta)
        j_star = B[k]

        x[j0] = theta0
        for i in range(len(B)):
            if i != k:
                x[B[i]] = x[B[i]] - theta0 * z[i]
        x[j_star] = 0

        B[k] = j0

    raise RuntimeError("Превышено максимальное число итераций!")


def initial_phase_simplex(c, A, b, max_iter=1000, eps=1e-9, forbidden_pairs=None):
    A_work = np.array(A, dtype=float).copy()
    b_work = np.array(b, dtype=float).copy()
    c = np.array(c, dtype=float).copy()

    m, n = A_work.shape

    for i in range(m):
        if b_work[i] < 0:
            b_work[i] *= -1
            A_work[i, :] *= -1

    I_m = np.eye(m)
    A_tilde = np.hstack([A_work, I_m])
    c_tilde = np.hstack([np.zeros(n), -np.ones(m)])

    x_tilde0 = np.zeros(n + m)
    x_tilde0[n:] = b_work
    B = list(range(n, n + m))

    status, x_tilde_opt, B = simplex_main_phase(
        c_tilde, A_tilde, x_tilde0, B, max_iter=max_iter,
        forbidden_pairs=forbidden_pairs
    )

    if status != "optimal":
        return {
            "feasible": False,
            "message": "Вспомогательная задача не решена до оптимума.",
            "x": None, "B": None, "A_reduced": None, "b_reduced": None,
        }

    if np.any(x_tilde_opt[n:] > eps):
        return {
            "feasible": False,
            "message": "Исходная задача несовместна (допустимых планов нет).",
            "x": None, "B": None, "A_reduced": None, "b_reduced": None,
        }

    x = x_tilde_opt[:n].copy()

    while any(j >= n for j in B):
        artificial_positions = [idx for idx, val in enumerate(B) if val >= n]
        k = max(artificial_positions, key=lambda idx: B[idx])
        j_k = B[k]

        A_B = A_tilde[:, B]
        A_B_inv = np.linalg.inv(A_B)

        nonbasic_real = [j for j in range(n) if j not in B]
        replacement = None
        for j in nonbasic_real:
            l_j = A_B_inv @ A_tilde[:, j]
            if abs(l_j[k]) > eps:
                replacement = j
                break

        if replacement is not None:
            B[k] = replacement
            continue

        col = A_tilde[:, j_k]
        candidate_rows = np.where(np.abs(col) > eps)[0]
        if len(candidate_rows) == 0:
            row_to_delete = k
        else:
            row_to_delete = int(candidate_rows[0])

        A_work = np.delete(A_work, row_to_delete, axis=0)
        b_work = np.delete(b_work, row_to_delete, axis=0)
        A_tilde = np.delete(A_tilde, row_to_delete, axis=0)
        B.pop(k)

        if len(B) == 0:
            break

    return {
        "feasible": True,
        "message": "Исходная задача совместна.",
        "x": x, "B": B, "A_reduced": A_work, "b_reduced": b_work,
    }

In [3]:
def wolf_method(c_orig, Q, A, b, max_iter=1000, eps=1e-9):
    """
    Метод Вульфа — решение задачи квадратичного программирования:

        min  c^T x + 1/2 * x^T Q x
        s.t. Ax = b,  x >= 0

    Q — симметричная положительно определённая матрица (n x n).

    По теореме Баранкина-Дорфмана план x оптимален тогда и только тогда,
    когда существуют v >= 0, u такие, что:
        Ax = b
        Qx - v + A^T u = -c
        v^T x = 0   (условие дополняющей нежёсткости)

    Замена u = u+ - u- (u+, u- >= 0) приводит к системе A_tilde * x_tilde = b_tilde,
    которую решаем начальной фазой симплекс-метода с запретом одновременного
    нахождения в базисе пар (i, n+i) — x_i и v_i.

    Вектор переменных x_tilde (размер 2n + 2m):
        ( x_1..x_n | v_1..v_n | u+_1..u+_m | u-_1..u-_m )

    Расширенная матрица (размер (m+n) x (2n+2m)):
        A_tilde = [ A,   0,    0,    0  ]
                  [ Q,  -I_n, A^T, -A^T ]
        b_tilde = [ b; -c ]
    """
    c_orig = np.array(c_orig, dtype=float)
    Q      = np.array(Q,      dtype=float)
    A      = np.array(A,      dtype=float)
    b      = np.array(b,      dtype=float)

    m, n = A.shape

    A_tilde = np.block([
        [A,  np.zeros((m, n)), np.zeros((m, m)), np.zeros((m, m))],
        [Q, -np.eye(n),        A.T,              -A.T             ],
    ])
    b_tilde = np.concatenate([b, -c_orig])

    # Запрещённые пары: x_i и v_i = x_{n+i} не могут быть одновременно в базисе
    forbidden_pairs = [(i, n + i) for i in range(n)]

    result = initial_phase_simplex(
        c=np.zeros(2 * n + 2 * m),
        A=A_tilde,
        b=b_tilde,
        max_iter=max_iter,
        eps=eps,
        forbidden_pairs=forbidden_pairs,
    )

    if not result["feasible"]:
        return {"status": "infeasible", "message": result["message"], "x": None, "f": None}

    x_opt = result["x"][:n]
    f_val = float(c_orig @ x_opt + 0.5 * x_opt @ Q @ x_opt)
    return {"status": "optimal", "message": "Оптимальное решение найдено.", "x": x_opt, "f": f_val}


# Пример 9 из лабораторной
# f(x) = x1^2 + 2x2^2 - 2x1x2 - 2x1 - 6x2 -> min
# x1 + x2 = 2,  -x1 + 2x2 = 2,  x1,x2 >= 0
# Ожидаемый ответ: x* = (2/3, 4/3),  f* = -64/9 ≈ -7.1111
if __name__ == "__main__":
    Q = np.array([[2, -2], [-2, 4]], dtype=float)
    c = np.array([-2, -6], dtype=float)
    A = np.array([[1, 1], [-1, 2]], dtype=float)
    b = np.array([2, 2], dtype=float)

    res = wolf_method(c, Q, A, b)
    print("Статус :", res["status"])
    print("x*     :", res["x"])
    print("f(x*)  :", res["f"])
    print()
    print("Ожидаемое x*  =", [2/3, 4/3])
    print("Ожидаемое f*  =", -64/9)

Статус : optimal
x*     : [0.66666667 1.33333333]
f(x*)  : -7.111111111111112

Ожидаемое x*  = [0.6666666666666666, 1.3333333333333333]
Ожидаемое f*  = -7.111111111111111
